# Class - 1(29-July-2026)

In [6]:
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


In [7]:
DATA_DIR = Path("/Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data")

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

PDF found:
/Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf


In [8]:
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Total PDF pages loaded: 77


In [9]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez 

In [10]:
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

In [11]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

In [12]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 's

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


In [14]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [15]:
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

In [12]:
eembeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [13]:
test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [24]:
## Using below emedigngs to create a vector database using Chroma as openAI getting error
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

/var/folders/s8/n1q5x4p510scnb04xgbb48j80000gn/T/ipykernel_2107/3754719291.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [25]:
test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 384
First 10 values: [-0.03594137355685234, 0.041557688266038895, -0.0662124902009964, 0.02400517836213112, -0.05041997879743576, -0.003377891145646572, -0.06243664026260376, -0.007374073378741741, 0.029075050726532936, 0.0783664882183075]


In [27]:
PERSIST_DIRECTORY = (
    Path("/Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/06_Retriever")
    / "chroma_llama2_retriever"
)
# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

In [28]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llama2_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 343
Persisted at: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/06_Retriever/chroma_llama2_retriever


In [29]:
# Load the existing Chroma collection
vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

print("Existing vector store loaded successfully.")
print(f"Persist directory: {PERSIST_DIRECTORY}")

Existing vector store loaded successfully.
Persist directory: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/06_Retriever/chroma_llama2_retriever


In [32]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

In [33]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [34]:
query = "What model sizes of Llama 2 were released?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-341
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
nor the fine-tuning datasets include Meta user data.
Data Freshness The pretraining data has a cutoff of September 2022, but some tuning data is
more recent, up to July 2023.
Evaluation Results
See evaluations for pretraining (Section 2); fine-tuning (Section 3); and safety (Section 4).
Ethical Considerations and Limitations(Section 5.2)
Llama 2 is a new technology that carries risks with use. Testing conducted to date has been in
English, and has not covered, nor could it cover all scenarios. For these reasons, as with all LLMs,
Llama 2’s potential outputs cannot be predicted in advance, and the model may in some instances
produce inaccurate or objectionable responses to user prompts. There

RANK: 2
PAPER PAGE: 77
SEC

In [35]:
## MMR Retriever
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [36]:
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 31
SECTION: safety
CHUNK ID: llama2-page-31-chunk-135
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
manceissimilaracrosscategories, Llama 2-Chathasrelativelymoreviolationsunderthe unqualifiedadvice
category (although still low in an absolute sense), for various reasons, including lack of an appropriate
disclaimer (e.g.,“I am not a professional”) at times. For the other two categories,Llama 2-Chat achieves
comparable or lower violation percentage consistently regardless of model sizes.
Truthfulness, Toxicity, and Bias.In Table 14, fine-tunedLlama 2-Chat shows great improvement over
the pretrainedLlama 2 in terms of truthfulness (50.18→ 64.14 for 70B) and toxicity (24.60→ 0.01 for 70B).
The percentage of toxic generations shrinks to effectively 0% forLlama 2-Chat of all sizes: this is the lo

RANK: 2
PAPER PAGE: 3
SECTIO

In [37]:
## Comparison between similarity and MMR retrievers

query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [38]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
31 safety llama2-page-31-chunk-135
3 introduction llama2-page-3-chunk-10
20 safety llama2-page-20-chunk-85
34 discussion llama2-page-34-chunk-146

MMR results:
31 safety llama2-page-31-chunk-135
3 introduction llama2-page-3-chunk-10
36 conclusion llama2-page-36-chunk-160
8 fine_tuning llama2-page-8-chunk-29


In [41]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.50,
    }
)

In [42]:
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 2
SECTION: front_matter
CHUNK ID: llama2-page-2-chunk-4
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
4 Safety 20
4.1 Safety in Pretraining . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 20
4.2 Safety Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 23
4.3 Red Teaming . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 28
4.4 Safety Evaluation of Llama 2-Chat . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 29
5 Discussion 32
5.1 Learnings and Observations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 32
5.2 Limitations and Ethical Considerations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 34
5.3 Res

RANK: 2
PAPER PAGE: 4
SECT

In [43]:
## Score from the retriever
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.6230
Paper page: 2
Section: front_matter
4 Safety 20
4.1 Safety in Pretraining . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 20
4.2 Safety Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 23
4.3 Red Teaming . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 28
4.4 Safety Evaluation of Llama 2-Chat . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 29
5 Discussion 32
5.1 Learnings and Observa
Rank: 2
Relevance score: 0.5936
Paper page: 4
Section: introduction
Solaiman et al., 2023). Testing conducted to date has been in English and has not — and could not — cover
all scenarios. Therefore, before deploying any applications ofLlama 2-Chat, developers should perform
safety testing and tuning tailored to their specific applications of the model. We provide a responsible use
guide¶ and code examples‖ to facilitate the sa

In [44]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


In [45]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (384,)
Document-vectors shape: (6, 384)


In [46]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [47]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,9,fine_tuning,llama2-page-9-chunk-34,0.468164,1.031344,0.468164,"learning rate of2× 10−5, a weight decay of 0.1..."
1,5,pretraining,llama2-page-5-chunk-14,0.458798,1.040386,0.458798,Figure 4: Training ofLlama 2-Chat: This proces...
2,32,discussion,llama2-page-32-chunk-137,0.417908,1.078973,0.417908,"5 Discussion Here, we discuss the interesting ..."
3,32,discussion,llama2-page-32-chunk-138,0.410382,1.085926,0.410382,"bility, seemed a somewhat shadowy field for th..."
4,10,fine_tuning,llama2-page-10-chunk-35,0.403929,1.091852,0.403929,"sampled human preferences, whereby human annot..."
5,11,fine_tuning,llama2-page-11-chunk-42,0.403557,1.092193,0.403557,Each example consists of a prompt (including p...


In [48]:
metric_query

'How was reinforcement learning with human feedback used?'

In [49]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,9,fine_tuning,0.468164,"learning rate of2× 10−5, a weight decay of 0.1..."
1,5,pretraining,0.458798,Figure 4: Training ofLlama 2-Chat: This proces...
2,32,discussion,0.417908,"5 Discussion Here, we discuss the interesting ..."
3,32,discussion,0.410382,"bility, seemed a somewhat shadowy field for th..."
4,10,fine_tuning,0.403929,"sampled human preferences, whereby human annot..."
5,11,fine_tuning,0.403557,Each example consists of a prompt (including p...


In [50]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,9,fine_tuning,1.031344,"learning rate of2× 10−5, a weight decay of 0.1..."
1,5,pretraining,1.040386,Figure 4: Training ofLlama 2-Chat: This proces...
2,32,discussion,1.078973,"5 Discussion Here, we discuss the interesting ..."
3,32,discussion,1.085926,"bility, seemed a somewhat shadowy field for th..."
4,10,fine_tuning,1.091852,"sampled human preferences, whereby human annot..."
5,11,fine_tuning,1.092193,Each example consists of a prompt (including p...


In [51]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,9,fine_tuning,0.468164,"learning rate of2× 10−5, a weight decay of 0.1..."
1,5,pretraining,0.458798,Figure 4: Training ofLlama 2-Chat: This proces...
2,32,discussion,0.417908,"5 Discussion Here, we discuss the interesting ..."
3,32,discussion,0.410382,"bility, seemed a somewhat shadowy field for th..."
4,10,fine_tuning,0.403929,"sampled human preferences, whereby human annot..."
5,11,fine_tuning,0.403557,Each example consists of a prompt (including p...


In [52]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.46816439 0.45879835 0.41790833 0.41038239 0.40392904 0.40355673]

Dot product after normalization:
[0.46816439 0.45879835 0.41790833 0.41038239 0.40392904 0.40355673]

Are they approximately equal? True


In [53]:
## Metadata filtering
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "fine_tuning"
        },
    }
)

In [54]:
query = "How was Llama 2-Chat aligned with human preferences?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: llama2-page-11-chunk-45
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
this study, the role of reward signals is to learn human preference forLlama 2-Chat outputs rather than
any modeloutputs. However, in our experiments, we do not observe negative transfer from the open-source
preference datasets. Thus, we have decided to keep them in our data mixture, as they could enable better
generalization for the reward model and prevent reward hacking, i.e.Llama 2-Chat taking advantage of
some weaknesses of our reward, and so artificially inflating the score despite performing less well.
With training data available from different sources, we experimented with different mixing recipes for both
Helpfulness and Safety reward models to ascertain the best settings. After ex

RANK: 2
PAPER PAGE: 10
S

In [55]:
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"

print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.


In [56]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

In [57]:
query = "How was human preference data collected?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: llama2-page-11-chunk-41
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Dataset
Num. of
Comparisons
Avg. # Turns
per Dialogue
Avg. # Tokens
per Example
Avg. # Tokens
in Prompt
Avg. # Tokens
in Response
Anthropic Helpful 122,387 3.0 251.5 17.7 88.4
Anthropic Harmless 43,966 3.0 152.5 15.7 46.4
OpenAI Summarize 176,625 1.0 371.1 336.0 35.1
OpenAI WebGPT 13,333 1.0 237.2 48.3 188.9
StackExchange 1,038,480 1.0 440.2 200.1 240.2
Stanford SHP 74,882 1.0 338.3 199.5 138.8
Synthetic GPT-J 33,139 1.0 123.3 13.0 110.3
Meta (Safety & Helpfulness) 1,418,091 3.9 798.5 31.4 234.1
Total 2,919,326 1.6 595.7 108.2 216.9
Table 6: Statistics of human preference data for reward modeling.We list both the open-source and
internally collected human preference data used for reward mode

RANK: 2
PAPER PAGE: 10
S

In [58]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 17
SECTION: fine_tuning
CHUNK ID: llama2-page-17-chunk-74
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
answers based on a 7-point Likert scale (the higher the better). We observe that our reward models overall
are well calibrated with our human preference annotations, as illustrated in Figure 29 in the appendix. This
confirms the relevance of using our reward as a point-wise metric, despite being trained with a Pairwise
Ranking Loss.
Still, as Goodhart’s Law states, when a measure becomes a target, it ceases to be a good measure. To ensure
ourmeasurewon’tdivergefromthehumanpreferences,weadditionallyusedamoregeneralreward,trained
17

RANK: 2
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-40
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
-----

In [60]:
## Post-Filtering
unfiltered_candidates = vector_store.similarity_search(
    query="How was the reward model trained?",
    k=15,
)

post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 17
SECTION: fine_tuning
CHUNK ID: llama2-page-17-chunk-74
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
answers based on a 7-point Likert scale (the higher the better). We observe that our reward models overall
are well calibrated with our human preference annotations, as illustrated in Figure 29 in the appendix. This
confirms the relevance of using our reward as a point-wise metric, despite being trained with a Pairwise
Ranking Loss.
Still, as Goodhart’s Law states, when a measure becomes a target, it ceases to be a good measure. To ensure
ourmeasurewon’tdivergefromthehumanpreferences,weadditionallyusedamoregeneralreward,trained
17

RANK: 2
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-40
SOURCE: /Users/rakhi/Documents/AI/AgenticAI/Practical/10_Rag/04_Vector_DB/data/llama2-research-paper.pdf
-----

In [61]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 10


# Class - 2(05-Aug-2026)

## Sparce,dence and hybrid retriever

In [ ]:
## Sparce and Dense retriever 

import os
from typing import List

from pydantic import BaseModel, Field


from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

/var/folders/s8/n1q5x4p510scnb04xgbb48j80000gn/T/ipykernel_2107/2688584358.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
/Users/rakhi/Documents/AI/AgenticAI/Practical/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [3]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

In [4]:
from langchain_community.retrievers import BM25Retriever

In [16]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [17]:
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x133a23b50>)

In [19]:
# Final number of results
bm25_retriever.k = 4

In [20]:
sparse_query = "Grouped-Query Attention GQA 70B"

In [21]:
sparse_documents = bm25_retriever.invoke(sparse_query)

In [22]:
display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)


Sparse Retrieval: BM25 Results

RANK: 1
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0× 10−4
13B 2k ✗ 1.0T 3.0× 10−4
33B 2k ✗ 1.4T 1.5× 10−4
65B 2k ✗ 1.4T 1.5× 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0× 10−4
13B 4k ✗ 2.0T 3.0× 10−4
34B 4k ✓ 2.0T 1.5× 10−4
70B 4k ✓ 2.0T 1.5× 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for
improved inference scalability.
0 250 500 750 1000 1250 1500 1750 

RANK: 2
Paper page: 48
Section: appendix
Chunk ID: llama2-page-48-chunk-220
----------------------------------------------------------------------------------------------------
BoolQ PIQA 

In [30]:
### Dense retrieval using the vector store 

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [31]:
dense_query = (
    "How did Meta improve inference scalability "
    "for the largest Llama 2 models?"
)

dense_documents = dense_retriever.invoke(dense_query)

In [32]:
display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 35
Section: discussion
Chunk ID: llama2-page-35-chunk-154
----------------------------------------------------------------------------------------------------
this progression is the rise of Llama, recognized for its focus on computational efficiency during inference
(Touvron et al., 2023). A parallel discourse has unfolded around the dynamics of open-source versus closed-
source models. Open-source releases like BLOOM (Scao et al., 2022), OPT(Zhang et al., 2022), and Falcon
(Penedo et al., 2023) have risen to challenge their closed-source counterparts like GPT-3 and Chinchilla.
§§https://ai.meta.com/llama
35

RANK: 2
Paper page: 77
Section: appendix
Chunk ID: llama2-page-77-chunk-341
----------------------------------------------------------------------------------------------------
nor the fine-tuning datasets include Meta user data.
Data Freshness The pretraining data has a cutoff of September 2022, but some tuning data is

In [33]:
# Comparison between sparse and dense retrievers

comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow 

In [34]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 4 llama2-page-4-chunk-12
2 47 llama2-page-47-chunk-217
3 6 llama2-page-6-chunk-18
4 54 llama2-page-54-chunk-240

DENSE RESULTS
1 47 llama2-page-47-chunk-217
2 6 llama2-page-6-chunk-18
3 4 llama2-page-4-chunk-12
4 48 llama2-page-48-chunk-220


In [35]:
# Hybrid retrieval
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

In [37]:
hybrid_query = (
    "Llama 2 70B grouped-query attention and inference scalability"
)
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

In [38]:
display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense + RRF

RANK: 1
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow significantly. For larger models, where KV cache size becomes a bottleneck,
key and value projections can be shared across multiple heads without much degradation of performance
(Chowdheryetal.,2022). Eithertheoriginalmulti-queryformatwithasingleKVprojection(MQA, Shazeer,
2019) or a grouped-query attention variant with 8 KV projections (GQA, Ainslie et al., 2023) can be used.
In Table 18, we compare MQA and GQA variants with an MHA baseline. We train all models with 150B
tokens while keeping a fixed 30B model size. To keep a similar overall parameter count across GQ

RANK: 2
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. L

                         ┌── BM25 Retriever ─────┐
User query ──────────────┤                       ├── Weighted RRF
                         └── Dense Retriever ────┘
                                                    ↓
                                            Combined ranking

In [39]:
test_query = (
    "How did Meta make Llama 2 70B efficient for large-scale inference?"
)

In [40]:
sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

In [41]:
display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 54
Section: appendix
Chunk ID: llama2-page-54-chunk-240
----------------------------------------------------------------------------------------------------
attribute, and so, up to 20 turns (we did not extend the human evaluation more, and all the examples had
less than 4048 tokens in total over the turns). As a comparison,Llama 2-Chat without GAtt can not anymore
refer to the attributes after only few turns: from 100% at turn t+1, to 10% at turn t+3 and then 0%.
GAtt Zero-shot Generalisation. We tried at inference time to set constrain not present in the training of
GAtt. For instance, “answer in one sentence only”, for which the model remained consistent, as illustrated in
Figure 28.
We applied first GAtt toLlama 1, which was pretrained with a 

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params C

### Query re-writing

In this Query rewriting, Query expansion and Query decomposition we transfor our query.

In [42]:
CHAT_MODEL = os.environ.get(
    "OPENAI_CHAT_MODEL",
    "gpt-4.1-mini",
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

Chat model: gpt-4.1-mini


In [43]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

In [44]:
query_rewriting_chain = (query_rewriting_prompt| llm | StrOutputParser())

In [ ]:
chat_history = """
                    User: How was Llama 2-Chat initially fine-tuned?
                    Assistant: It first underwent supervised fine-tuning.
                    """

original_query = "What did Meta do after that?"

rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

In [ ]:
print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

In [ ]:
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

In [ ]:
display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)

### Expanded Query

In [47]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

In [48]:
query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

In [49]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

In [50]:
query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

In [51]:
original_query = (
    "How was Llama 2-Chat improved using human feedback?"
)


In [ ]:
expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

In [ ]:
expanded_output

In [ ]:
all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]

In [ ]:
all_expanded_queries

In [ ]:
print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

In [ ]:
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)

### Multi Query Retrieve

In [ ]:
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever,
)

In [ ]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

In [ ]:
multi_query_documents = multi_query_retriever.invoke(
    "How did human feedback improve Llama 2-Chat?"
)

In [ ]:
display_documents(
    multi_query_documents,
    title="Built-in MultiQueryRetriever Results",
    max_documents=10,
)

### Query Decomposed:

class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description=(
            "Independent and atomic search queries required "
            "to answer the complete user question."
        )
    )

In [ ]:
query_decomposition_llm = llm.with_structured_output(
    DecomposedQueryOutput
)

In [ ]:
query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Break the user's complex question into independent,
atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.
""",
        ),
        (
            "human",
            "Complex query: {query}",
        ),
    ]
)

In [ ]:
query_decomposition_chain = ( query_decomposition_prompt | query_decomposition_llm)

In [ ]:
complex_query = """
Compare Llama 2 pretraining with Llama 2-Chat fine-tuning,
and explain how Meta improved model safety.
"""

In [ ]:
decomposed_output = query_decomposition_chain.invoke(
    {
        "query": complex_query
    }
)

sub_queries = decomposed_output.sub_queries

In [ ]:
print("Original complex query:")
print(complex_query)

print("\nGenerated sub-queries:")

for number, query in enumerate(sub_queries, start=1):
    print(f"{number}. {query}")

In [ ]:
decomposition_results = {}

for sub_query in sub_queries:
    decomposition_results[sub_query] = hybrid_retriever.invoke(
        sub_query
    )

In [ ]:
for sub_query, documents in decomposition_results.items():
    display_documents(
        documents,
        title=f"Sub-query: {sub_query}",
        max_documents=4,
    )

In [ ]:
all_decomposition_documents = []

for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(
    all_decomposition_documents
)

display_documents(
    all_decomposition_documents,
    title="Combined Evidence from All Sub-Queries",
    max_documents=12,
)

Complex query<br>
      ↓<br>
Atomic sub-queries<br>
      ↓<br>
Retrieve for every sub-query<br>
      ↓<br>
Merge evidence<br>
      ↓<br>
Remove duplicates

### Reranking

In [ ]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

In [ ]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

In [ ]:
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

In [ ]:
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
reranking_query = (
    "How did Meta collect and use human preference data to train Llama 2-Chat?"
)

In [ ]:
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

In [ ]:
initial_candidates

In [ ]:
display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)

In [ ]:
reranked_documents = reranking_retriever.invoke(reranking_query)

In [ ]:
display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)

User query<br>
     ↓<br>
Dense Retriever<br>
     ↓<br>
Top 20 candidates<br>
     ↓<br>
Cross-Encoder Reranker<br>
     ↓<br>
Query-document relevance evaluation<br>
     ↓<br>
Final top 5 documents<br>

In [55]:
## Reranking with BM25 and Dense Retriever 

bm25_retriever.k = 15

dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

In [56]:
hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

In [ ]:
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
query = (
    "What techniques did Meta use to improve the helpfulness and safety of Llama 2-Chat?"
)

final_documents = hybrid_reranking_retriever.invoke(query)

final_documents

In [ ]:
display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)

                         ┌── BM25 Search ───────┐
User query ──────────────┤                      ├── Weighted RRF
                         └── Dense Search ──────┘
                                                   ↓
                                          Candidate documents
                                                   ↓
                                       Cross-Encoder Reranker
                                                   ↓
                                           Final top documents

Sparse Retrieval
    = BM25Retriever

Dense Retrieval
    = VectorStoreRetriever

Hybrid Retrieval
    = BM25 + Dense + EnsembleRetriever + Weighted RRF

Query Rewriting
    = Convert a contextual query into a standalone query

Query Expansion
    = Generate related query variations and merge their results

Query Decomposition
    = Split one complex query into atomic searchable queries

HyDE
    = Generate a hypothetical answer passage and use its embedding for dense retrieval

Reranking
    = Retrieve broad candidates and reorder them using a cross-encoder

1. BM25 sparse retrieval
2. Dense vector retrieval
3. Hybrid retrieval
4. Query rewriting
5. Query expansion
6. Query decomposition
7. HyDE retrieval
8. Dense retrieval + reranking
9. Hybrid retrieval + reranking

### HyDE (Hypothetical Document Embeddings)

HyDE asks an LLM to write a plausible passage that could answer the query. The hypothetical passage is embedded and used for dense retrieval instead of embedding the short user query directly. The generated text is only a search aid; the final answer must still be grounded in retrieved source documents.

In [ ]:
from typing import Any

from langchain_core.retrievers import BaseRetriever


class HyDERetriever(BaseRetriever):
    """Generate a hypothetical passage and retrieve real documents with it."""

    hypothesis_chain: Any
    vector_retriever: Any
    last_hypothetical_document: str = ""

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager,
    ):
        hypothetical_document = self.hypothesis_chain.invoke(
            {"query": query}
        ).strip()

        self.last_hypothetical_document = hypothetical_document

        return self.vector_retriever.invoke(
            hypothetical_document,
            config={
                "callbacks": run_manager.get_child(),
            },
        )

In [ ]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Write a concise research-paper passage that would directly answer the user's question.
Use technical terminology and named entities likely to appear in the source document.
Do not mention that the passage is hypothetical.
Do not add citations, headings, or commentary.
Return only the passage.
""",
        ),
        ("human", "Question: {query}"),
    ]
)

hyde_hypothesis_chain = (
    hyde_prompt
    | llm
    | StrOutputParser()
)

hyde_dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

hyde_retriever = HyDERetriever(
    hypothesis_chain=hyde_hypothesis_chain,
    vector_retriever=hyde_dense_retriever,
)

In [ ]:
hyde_query = (
    "How did Meta use human feedback to improve the helpfulness "
    "and safety of Llama 2-Chat?"
)

hyde_documents = hyde_retriever.invoke(hyde_query)

print("Original query:")
print(hyde_query)

print("\nHypothetical document used for retrieval:")
print(hyde_retriever.last_hypothetical_document)

display_documents(
    hyde_documents,
    title="HyDE Retrieval Results",
    max_documents=5,
)

User query<br>
     ↓<br>
LLM-generated hypothetical document<br>
     ↓<br>
Embed the hypothetical document<br>
     ↓<br>
Dense vector search<br>
     ↓<br>
Real source documents<br>

### HyDE with LangChain's built-in helper

`HypotheticalDocumentEmbedder` is an embeddings wrapper, not a complete retriever. For a query, it generates a hypothetical document with the LLM and embeds that generated text. Chroma then uses the resulting vector to search the collection of real PDF chunks.

In [ ]:
try:
    # LangChain v1: legacy chains live in langchain-classic.
    from langchain_classic.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )
except ImportError:
    # Compatibility fallback for older LangChain versions.
    from langchain.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )

In [ ]:
built_in_hyde_embeddings = (
    HypotheticalDocumentEmbedder.from_llm(
        llm=llm,
        base_embeddings=embeddings,
        prompt_key="web_search",
    )
)

# Reuse the existing collection. Real PDF chunks remain embedded with
# `embeddings`; HyDE is used only to transform query embeddings.
built_in_hyde_vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=built_in_hyde_embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

built_in_hyde_retriever = (
    built_in_hyde_vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5},
    )
)

In [ ]:
built_in_hyde_query = (
    "How did Meta use human feedback to improve the helpfulness "
    "and safety of Llama 2-Chat?"
)

built_in_hyde_documents = (
    built_in_hyde_retriever.invoke(
        built_in_hyde_query
    )
)

display_documents(
    built_in_hyde_documents,
    title="Built-in LangChain HyDE Results",
    max_documents=5,
)

> **Note:** Do not use `built_in_hyde_embeddings` when initially indexing the PDF chunks. Index real documents with the normal `OpenAIEmbeddings` object, then use the HyDE wrapper only when loading the collection for retrieval.